# Sensitivity of the SHAP Core-Stable Gene Set



## 1. Functions

In [1]:
import argparse
import os
import time
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

TUNED_C = 6.581332043291798        # "Best C for LogisticRegression" printed by the notebook
PAPER_R, PAPER_K, PAPER_THRESHOLD = 20, 20, 0.8
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
RESAMPLES = [10, 20, 30, 40, 50]
TOP_K = [10, 15, 20, 30, 50]


def load_primary_tumor(path):
    expr = pd.read_csv(path, index_col=0).T
    expr = expr[expr.index.str.endswith("-01")]
    return expr.astype(np.float32)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="../data")
    ap.add_argument("--out", default="../results/shap_sensitivity")
    ap.add_argument("--reference", default=os.path.join("../results/final_pipeline", "shap_gene_stability.csv"),
                    help="stability table saved by the notebook, used for the reproduction check")
    args = ap.parse_args(argv)
    os.makedirs(args.out, exist_ok=True)
    n_max = max(RESAMPLES)

    # ---------------- data: identical to the notebook ----------------
    X_luad = load_primary_tumor(os.path.join(args.data_dir, "LUAD.csv"))
    X_lusc = load_primary_tumor(os.path.join(args.data_dir, "LUSC.csv"))
    common_genes = X_luad.columns.intersection(X_lusc.columns)
    X = pd.concat([X_luad[common_genes], X_lusc[common_genes]], axis=0)
    y_bin = np.array([0] * len(X_luad) + [1] * len(X_lusc))
    X_vals = X.values
    gene_names = np.array([str(g).split("|")[0] for g in X.columns])
    print(f"Data: {X_vals.shape[0]} samples, {X_vals.shape[1]} genes | shap {shap.__version__}", flush=True)

    # ---------------- SHAP over n_max resamples (with resume) ----------------
    imp_path = os.path.join(args.out, "mean_abs_shap_per_resample.npy")
    imp = np.load(imp_path) if os.path.exists(imp_path) else np.full((n_max, X_vals.shape[1]), np.nan, np.float32)
    t0 = time.time()
    for seed in range(n_max):
        if not np.isnan(imp[seed, 0]):
            continue
        Xtr_s, Xtemp_s, ytr_s, ytemp_s = train_test_split(
            X_vals, y_bin, test_size=0.2, random_state=seed, stratify=y_bin)
        Xva_s, Xte_s, yva_s, yte_s = train_test_split(
            Xtemp_s, ytemp_s, test_size=0.5, random_state=seed, stratify=ytemp_s)
        m = LogisticRegression(C=TUNED_C, max_iter=3000, class_weight="balanced")
        m.fit(Xtr_s, ytr_s)
        explainer_s = shap.LinearExplainer(m, Xtr_s)
        shap_vals_s = explainer_s.shap_values(Xte_s)
        imp[seed] = np.abs(shap_vals_s).mean(axis=0)
        np.save(imp_path, imp)
        print(f"  resample {seed + 1}/{n_max} done ({time.time() - t0:.0f}s)", flush=True)

    def top_list(seed, k):
        return gene_names[np.argsort(imp[seed])[::-1][:k]]

    def frequencies(R, K):
        c = Counter()
        for s in range(R):
            c.update(set(top_list(s, K)))
        return pd.Series({g: n / R for g, n in c.items()}).sort_values(ascending=False)

    def core_set(R, K, t):
        f = frequencies(R, K)
        return set(f[f >= t - 1e-9].index)

    lines = []
    baseline = core_set(PAPER_R, PAPER_K, PAPER_THRESHOLD)
    f20 = frequencies(PAPER_R, PAPER_K)
    f20.rename("stability").to_csv(os.path.join(args.out, "stability_R20_K20.csv"), header=True)

    # ---------------- 1. reproduction check ----------------
    lines.append("=== 1. Reproduction check (resamples 1-20 vs the notebook) ===")
    if os.path.exists(args.reference):
        ref = pd.read_csv(args.reference)
        ref = pd.Series(ref["stability_pct"].values / 100, index=ref["gene"].astype(str))
        ref_core = set(ref[ref >= PAPER_THRESHOLD - 1e-9].index)
        genes = sorted(set(ref.index) | set(f20.index))
        diff = (ref.reindex(genes).fillna(0) - f20.reindex(genes).fillna(0)).abs()
        lines.append(f"Core set identical to the notebook: {baseline == ref_core} "
                     f"({len(baseline)} genes here, {len(ref_core)} in the notebook)")
        lines.append(f"Largest difference in any gene's stability: {diff.max():.2f}")
        if baseline != ref_core:
            lines.append(f"  only here: {sorted(baseline - ref_core)} | only in notebook: {sorted(ref_core - baseline)}")
    else:
        lines.append(f"Reference file not found ({args.reference}); check skipped.")
    lines.append(f"Baseline core set (R=20, top 20, >=0.8): {len(baseline)} genes: {sorted(baseline)}")

    # ---------------- 2. sensitivity table ----------------
    rows = []
    for setting, values in [("threshold", THRESHOLDS), ("resamples_R", RESAMPLES), ("top_K", TOP_K)]:
        for v in values:
            R, K, t = PAPER_R, PAPER_K, PAPER_THRESHOLD
            if setting == "threshold":
                t = v
            elif setting == "resamples_R":
                R = v
            else:
                K = v
            c = core_set(R, K, t)
            rows.append({"setting": setting, "value": v, "n_core_genes": len(c),
                         "baseline_genes_kept": len(c & baseline),
                         "dropped": ", ".join(sorted(baseline - c)),
                         "added": ", ".join(sorted(c - baseline)),
                         "jaccard_vs_baseline": round(len(c & baseline) / len(c | baseline), 3) if c | baseline else 1.0})
    sens = pd.DataFrame(rows)
    sens.to_csv(os.path.join(args.out, "core_set_sensitivity.csv"), index=False)
    lines.append("\n=== 2. Core-set sensitivity (one setting varied at a time) ===")
    lines.append(sens.to_string(index=False))

    # ---------------- 3. rank stability of the baseline genes over all resamples ----------------
    ranks = np.argsort(np.argsort(-imp, axis=1), axis=1) + 1
    gi = {g: i for i, g in enumerate(gene_names)}
    rank_df = pd.DataFrame([{"gene": g,
                             "median_rank": int(np.median(ranks[:, gi[g]])),
                             "best_rank": int(ranks[:, gi[g]].min()),
                             "worst_rank": int(ranks[:, gi[g]].max()),
                             f"share_in_top20_of_{n_max}": round(float(np.mean(ranks[:, gi[g]] <= 20)), 2)}
                            for g in baseline]).sort_values("median_rank")
    rank_df.to_csv(os.path.join(args.out, "core_gene_ranks.csv"), index=False)
    lines.append(f"\n=== 3. Ranks of the baseline core genes over {n_max} resamples (of {len(gene_names)} genes) ===")
    lines.append(rank_df.to_string(index=False))

    # ---------------- 4. agreement of the top-20 lists ----------------
    lists = [set(top_list(s, PAPER_K)) for s in range(n_max)]
    jac = [len(lists[i] & lists[j]) / len(lists[i] | lists[j])
           for i in range(n_max) for j in range(i + 1, n_max)]
    lines.append(f"\n=== 4. Pairwise Jaccard index of the top-20 lists over {n_max} resamples ===")
    lines.append(f"mean {np.mean(jac):.3f}, median {np.median(jac):.3f}, min {np.min(jac):.3f}, max {np.max(jac):.3f}")

    report = "\n".join(lines)
    print("\n" + report)
    with open(os.path.join(args.out, "shap_sensitivity_report.txt"), "w", encoding="utf-8") as f:
        f.write(report + "\n")
    print(f"\nSaved to {args.out}/")

## 2. Run
Results: `results_shap_sensitivity/`

In [2]:
main(["--data-dir", "../data", "--out", "../results/shap_sensitivity"])

Data: 1017 samples, 20530 genes | shap 0.51.0
  resample 1/50 done (2s)
  resample 2/50 done (5s)
  resample 3/50 done (7s)
  resample 4/50 done (9s)
  resample 5/50 done (12s)
  resample 6/50 done (14s)
  resample 7/50 done (16s)
  resample 8/50 done (18s)
  resample 9/50 done (20s)
  resample 10/50 done (23s)
  resample 11/50 done (26s)
  resample 12/50 done (29s)
  resample 13/50 done (30s)
  resample 14/50 done (33s)
  resample 15/50 done (36s)
  resample 16/50 done (38s)
  resample 17/50 done (41s)
  resample 18/50 done (42s)
  resample 19/50 done (44s)
  resample 20/50 done (47s)
  resample 21/50 done (49s)
  resample 22/50 done (51s)
  resample 23/50 done (53s)
  resample 24/50 done (56s)
  resample 25/50 done (58s)
  resample 26/50 done (60s)
  resample 27/50 done (63s)
  resample 28/50 done (64s)
  resample 29/50 done (66s)
  resample 30/50 done (69s)
  resample 31/50 done (71s)
  resample 32/50 done (73s)
  resample 33/50 done (76s)
  resample 34/50 done (78s)
  resample 35/5